In [ ]:
import pandas as pd
from collections import Counter
from tqdm import tqdm
import stanza


# ---------- CONFIG ----------
PATH_CHUNKS = r"..\data\results\chunks_etiquetados_binario.xlsx"

OUTPUT_FIGS = r"..\reports\ideas PAPERS\figures"
OUTPUT_RESULTS = r"..\data\results"

# ---------- CARGAR DATOS ----------
chunks_etiquetados = pd.read_excel(PATH_CHUNKS)



In [ ]:
chunks_etiquetados.head()

,chunk_id,id_doc,texto_chunk,Ciencias_ambientales_ingenieria,Ciencias_espacio,Ciencias_fisicas,Ciencias_Geografia_oceanografia,Ciencias_medicas,Ciencias_metereologia,Ciencias_naturales,...,Ciencias_quimicas,Ciencias_tierra,Ciencia_Administracion_ciencia_investigacion,Ciencia_biologia,Ciencia_enfoque_cientifico,Ciencia_hidrologia,Ciencia_matematicas_estadistica,Ciencia_patologia,Ciencia_recursos_naturales,categorias_detectadas
0,1,6,le reconoce a las ideas de esas mismas persona...,0.465540,0.248854,0.204274,0.476426,0.301254,0.374194,0.446099,...,0.163613,0.415012,0.398073,0.313155,0.396260,0.346799,0.283724,0.356982,0.442019,"[('Ciencias_ambientales_ingenieria', 0.4655396..."
1,0,11,El método es sencillo. Primero tensar un poco ...,0.356047,0.317830,0.238794,0.326505,0.237563,0.329778,0.292988,...,0.115392,0.335479,0.351090,0.184611,0.386147,0.253733,0.305145,0.336977,0.273428,"[('Ciencias_polucion_catastrofes_seguridad', 0..."
2,1,29,otra cara de la moneda de Cali y la opinión pú...,0.343402,0.168502,0.146384,0.270736,0.275370,0.207571,0.237798,...,0.174043,0.245090,0.294449,0.162607,0.292631,0.254456,0.238961,0.300786,0.274115,"[('Ciencias_polucion_catastrofes_seguridad', 0..."
3,0,31,2018 será un año muy movido en ciberpolítica. ...,0.312686,0.223329,0.114333,0.242219,0.207143,0.269601,0.188867,...,0.097965,0.238194,0.423176,0.123846,0.311720,0.166820,0.264805,0.188973,0.265887,[('Ciencia_Administracion_ciencia_investigacio...
4,1,35,"comerán sus palabras de tiempos furiosos, fabr...",0.352538,0.226428,0.258161,0.265537,0.142906,0.347398,0.241312,...,0.236485,0.279226,0.184704,0.200993,0.187493,0.314507,0.109603,0.236698,0.370257,"[('Ciencias_polucion_catastrofes_seguridad', 0..."


In [7]:
# --- Filtrar el dataframe original ---
df_filtrado = chunks_etiquetados[['chunk_id', 'id_doc', 'texto_chunk', 
                                  'categorias_detectadas', 'etiqueta_ciencia']].copy()


In [5]:
!pip install stanza tqdm

Defaulting to user installation because normal site-packages is not writeable


In [6]:
# --- Inicializar el pipeline de Stanza ---
nlp_stanza = stanza.Pipeline(
    lang="es",
    processors="tokenize,pos,lemma",
    use_gpu=True  # o False si no tienes GPU
)


2025-12-11 04:21:40 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2025-12-11 04:21:40 INFO: Downloaded file to C:\Users\karen\stanza_resources\resources.json
2025-12-11 04:21:40 WARNING: Language es package default expects mwt, which has been added
2025-12-11 04:21:41 INFO: Loading these models for language: es (Spanish):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2025-12-11 04:21:41 WARNING: GPU requested, but is not available!
2025-12-11 04:21:41 INFO: Using device: cpu
2025-12-11 04:21:41 INFO: Loading: tokenize
2025-12-11 04:21:41 INFO: Loading: mwt
2025-12-11 04:21:41 INFO: Loading: pos
2025-12-11 04:21:44 INFO: Loading: lemma
2025-12-11 04:21:46 INFO: Done loading processors!


# Extraer frecuencias de POS a los textos

In [ ]:
def extraer_pos_frecuencias_stanza(textos):
    """
    Procesa una lista de textos con Stanza y devuelve tres listas de diccionarios:
      - verbos_frec[i]: {lema: frecuencia}
      - adjetivos_frec[i]: {lema: frecuencia}
      - sustantivos_frec[i]: {lema: frecuencia}
    """
    verbos_frec, adjetivos_frec, sustantivos_frec = [], [], []

    for texto in tqdm(textos, desc="Procesando con Stanza"):
        try:
            if not isinstance(texto, str) or not texto.strip():
                verbos_frec.append({})
                adjetivos_frec.append({})
                sustantivos_frec.append({})
                continue
                
            doc = nlp_stanza(texto)
            verbos, adjetivos, sustantivos = [], [], []

            for sent in doc.sentences:
                for w in sent.words:
                    if not w.lemma.isalpha():
                        continue
                    if w.upos == "VERB":
                        verbos.append(w.lemma.lower())
                    elif w.upos == "ADJ":
                        adjetivos.append(w.lemma.lower())
                    elif w.upos == "NOUN":
                        sustantivos.append(w.lemma.lower())

            verbos_frec.append(dict(Counter(verbos)))
            adjetivos_frec.append(dict(Counter(adjetivos)))
            sustantivos_frec.append(dict(Counter(sustantivos)))

        except Exception as e:
            print(f"Error procesando texto: {e}")
            verbos_frec.append({})
            adjetivos_frec.append({})
            sustantivos_frec.append({})

    return verbos_frec, adjetivos_frec, sustantivos_frec

In [ ]:
# --- Obtener listas de frecuencias ---
textos = df_filtrado['texto_chunk'].tolist()
verbos_frec, adjetivos_frec, sustantivos_frec = extraer_pos_frecuencias_stanza(textos)

# --- Crear DataFrames separados para cada categoría gramatical ---

# DataFrame para verbos
df_verbos = df_filtrado.copy()
df_verbos['verbos_lemas_frecuencias'] = verbos_frec
df_verbos.to_excel('verbos_stanza.xlsx', index=False)


=== Procesando df_chunks ===


Procesando con Stanza: 100%|██████████| 4287/4287 [2:23:49<00:00,  2.01s/it]  


In [ ]:
# DataFrame para adjetivos
df_adjetivos = df_filtrado.copy()
df_adjetivos['adjetivos_lemas_frecuencias'] = adjetivos_frec
df_adjetivos.to_excel('adjetivos_stanza.xlsx', index=False)

In [ ]:
# DataFrame para sustantivos
df_sustantivos = df_filtrado.copy()
df_sustantivos['sustantivos_lemas_frecuencias'] = sustantivos_frec
df_sustantivos.to_excel('sustantivos_stanza.xlsx', index=False)